In [1]:
!pip install -q --upgrade pip
!pip install -q pandas numpy pyarrow lightgbm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/energy-forecast"

feat = pd.read_parquet(f"{DRIVE_DIR}/features_hourly.parquet")

FEATURES = [
    "hour_of_day", "dow", "month", "is_weekend", "is_holiday",
    "lag_1", "lag_2", "lag_3", "lag_24", "lag_48", "lag_72", "lag_168",
    "roll_24_mean", "roll_24_std", "roll_168_mean",
    "temperature", "apparentTemperature", "humidity", "windSpeed",
]

def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

def evaluate(model, part, label):
    pred = model.predict(part[FEATURES], num_iteration=model.best_iteration)
    overall = mae(part["demand"], pred)
    hol = part[part["is_holiday"] == 1]
    hol_mae = mae(hol["demand"], model.predict(hol[FEATURES], num_iteration=model.best_iteration)) if len(hol) else float("nan")
    print(f"{label:28s} {part.name:>5s}  overall MAE {overall:.4f} | holiday MAE {hol_mae:.4f} (n={len(hol)})")
    return overall, hol_mae

Mounted at /content/drive


In [3]:
PARAMS = {
    "objective": "regression", "metric": "l2", "learning_rate": 0.05,
    "num_leaves": 63, "min_data_in_leaf": 50,
    "feature_fraction": 0.9, "bagging_fraction": 0.9, "bagging_freq": 1,
    "seed": 42, "verbose": -1,
}

def train_model(data):
    tr = data[data["hour"] < "2013-11-01"]
    va = data[(data["hour"] >= "2013-11-01") & (data["hour"] < "2014-01-01")]
    dtrain = lgb.Dataset(tr[FEATURES], label=tr["demand"])
    dval = lgb.Dataset(va[FEATURES], label=va["demand"], reference=dtrain)
    return lgb.train(PARAMS, dtrain, num_boost_round=3000,
                     valid_sets=[dval],
                     callbacks=[lgb.early_stopping(100, verbose=False)])

train = feat[feat["hour"] < "2013-11-01"].copy(); train.name = "train"
val = feat[(feat["hour"] >= "2013-11-01") & (feat["hour"] < "2014-01-01")].copy(); val.name = "val"
test = feat[feat["hour"] >= "2014-01-01"].copy(); test.name = "test"

m_base = train_model(feat)
print(f"baseline best_iter: {m_base.best_iteration}")
r_base = [evaluate(m_base, p, "original encoding") for p in (val, test)]

baseline best_iter: 468
original encoding              val  overall MAE 0.0129 | holiday MAE 0.0351 (n=48)
original encoding             test  overall MAE 0.0111 | holiday MAE 0.0262 (n=24)


In [4]:
feat_fix = feat.copy()
hol_mask = feat_fix["is_holiday"] == 1
feat_fix.loc[hol_mask, "dow"] = 6
feat_fix.loc[hol_mask, "is_weekend"] = 1
print(f"Rows re-encoded as Sunday: {int(hol_mask.sum()):,} "
      f"(train {int((feat_fix['hour'] < '2013-11-01') .__and__(hol_mask).sum())}, "
      f"test {int((feat_fix['hour'] >= '2014-01-01').__and__(hol_mask).sum())})")

val_fix = feat_fix[(feat_fix["hour"] >= "2013-11-01") & (feat_fix["hour"] < "2014-01-01")].copy(); val_fix.name = "val"
test_fix = feat_fix[feat_fix["hour"] >= "2014-01-01"].copy(); test_fix.name = "test"

m_fix = train_model(feat_fix)
print(f"fixed best_iter: {m_fix.best_iteration}")
r_fix = [evaluate(m_fix, p, "holiday-as-Sunday") for p in (val_fix, test_fix)]

Rows re-encoded as Sunday: 168 (train 96, test 24)
fixed best_iter: 506
holiday-as-Sunday              val  overall MAE 0.0125 | holiday MAE 0.0312 (n=48)
holiday-as-Sunday             test  overall MAE 0.0109 | holiday MAE 0.0208 (n=24)


In [5]:
rows = [
    {"encoding": "original", "split": "val", "overall_MAE": r_base[0][0], "holiday_MAE": r_base[0][1]},
    {"encoding": "holiday-as-Sunday", "split": "val", "overall_MAE": r_fix[0][0], "holiday_MAE": r_fix[0][1]},
    {"encoding": "original", "split": "test", "overall_MAE": r_base[1][0], "holiday_MAE": r_base[1][1]},
    {"encoding": "holiday-as-Sunday", "split": "test", "overall_MAE": r_fix[1][0], "holiday_MAE": r_fix[1][1]},
]
verdict = pd.DataFrame(rows)
print(verdict.to_string(index=False))

hol_val_delta = (r_base[0][1] - r_fix[0][1]) / r_base[0][1] * 100
overall_val_delta = (r_base[0][0] - r_fix[0][0]) / r_base[0][0] * 100
print(f"\nValidation holiday MAE change:  {-hol_val_delta:+.1f}% (positive = worse)")
print(f"Validation overall MAE change:  {-overall_val_delta:+.2f}%")
print()
print("Decision rule: adopt the fix if holiday MAE improves without hurting overall MAE.")

         encoding split  overall_MAE  holiday_MAE
         original   val     0.012878     0.035101
holiday-as-Sunday   val     0.012482     0.031202
         original  test     0.011134     0.026152
holiday-as-Sunday  test     0.010898     0.020766

Validation holiday MAE change:  -11.1% (positive = worse)
Validation overall MAE change:  -3.07%

Decision rule: adopt the fix if holiday MAE improves without hurting overall MAE.


In [6]:
def train_quantile_fix(alpha):
    params = dict(PARAMS)
    params.update({"objective": "quantile", "alpha": alpha, "metric": "quantile"})
    tr = feat_fix[feat_fix["hour"] < "2013-11-01"]
    va = feat_fix[(feat_fix["hour"] >= "2013-11-01") & (feat_fix["hour"] < "2014-01-01")]
    dtrain = lgb.Dataset(tr[FEATURES], label=tr["demand"])
    dval = lgb.Dataset(va[FEATURES], label=va["demand"], reference=dtrain)
    return lgb.train(params, dtrain, num_boost_round=3000, valid_sets=[dval],
                     callbacks=[lgb.early_stopping(100, verbose=False)])

q10_v2, q50_v2, q90_v2 = (train_quantile_fix(a) for a in (0.1, 0.5, 0.9))

va = feat_fix[(feat_fix["hour"] >= "2013-11-01") & (feat_fix["hour"] < "2014-01-01")]
resid = va["demand"] - q50_v2.predict(va[FEATURES], num_iteration=q50_v2.best_iteration)
lo_v2, hi_v2 = np.quantile(resid, [0.1, 0.9])
print(f"v2 conformal offsets: lo={lo_v2:+.4f}, hi={hi_v2:+.4f}")

te = feat_fix[feat_fix["hour"] >= "2014-01-01"]
p50_te = q50_v2.predict(te[FEATURES], num_iteration=q50_v2.best_iteration)
cov = ((te["demand"] >= p50_te + lo_v2) & (te["demand"] <= p50_te + hi_v2)).mean()
print(f"v2 conformal coverage on test: {cov:.1%}")

q10_v2.save_model(f"{DRIVE_DIR}/lightgbm_q10_v2.txt")
q50_v2.save_model(f"{DRIVE_DIR}/lightgbm_q50_v2.txt")
q90_v2.save_model(f"{DRIVE_DIR}/lightgbm_q90_v2.txt")
import json
with open(f"{DRIVE_DIR}/v2_intervals.json", "w") as f:
    json.dump({"lo": float(lo_v2), "hi": float(hi_v2), "coverage_test": float(cov)}, f, indent=2)
print("v2 quantile models + intervals saved to Drive")

v2 conformal offsets: lo=-0.0192, hi=+0.0180
v2 conformal coverage on test: 82.6%
v2 quantile models + intervals saved to Drive


In [7]:
import json

m_fix.save_model(f"{DRIVE_DIR}/lightgbm_demand_v2_holidayfix.txt")

experiment = {
    "change": "holiday-as-Sunday encoding (dow=6, is_weekend=1 for is_holiday rows)",
    "motivation": "Phase 4 error analysis: holiday MAE 2.4x baseline",
    "val_original": {"overall": r_base[0][0], "holiday": r_base[0][1]},
    "val_fixed": {"overall": r_fix[0][0], "holiday": r_fix[0][1]},
    "test_original": {"overall": r_base[1][0], "holiday": r_base[1][1]},
    "test_fixed": {"overall": r_fix[1][0], "holiday": r_fix[1][1]},
    "protocol_note": "Fix motivated by test-set error analysis; validation used as primary judge.",
}
with open(f"{DRIVE_DIR}/phase4b_holiday_fix.json", "w") as fh:
    json.dump(experiment, fh, indent=2)
print("Saved v2 point model + experiment record")


Saved v2 point model + experiment record
